In [2]:
from pathlib import Path
import shutil
 
import numpy as np
import pandas as pd
 
import mlflow
import mlflow.sklearn
import optuna
 
from mlflow import MlflowClient
from mlflow.models import infer_signature
 
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    TargetEncoder,
    PolynomialFeatures,
    KBinsDiscretizer
)

# Лабораторная работа №2
## Проведение экспериментов по настройке модели

In [4]:
CURRENT_DIR = Path.cwd()
 
if CURRENT_DIR.name == "research":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR
 
DATA_DIR = PROJECT_DIR / "data"
RESEARCH_DIR = PROJECT_DIR / "research"
MLFLOW_DIR = PROJECT_DIR / "mlflow"
 
DATA_PATH = DATA_DIR / "clean_dataset.pkl"
REQUIREMENTS_PATH = PROJECT_DIR / "requirements.txt"
 
RESEARCH_DIR.mkdir(exist_ok=True)
MLFLOW_DIR.mkdir(exist_ok=True)
 
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_PATH:", DATA_PATH)
print("RESEARCH_DIR:", RESEARCH_DIR)
print("MLFLOW_DIR:", MLFLOW_DIR)

PROJECT_DIR: /home/zhuk/ml_lab
DATA_PATH: /home/zhuk/ml_lab/data/clean_dataset.pkl
RESEARCH_DIR: /home/zhuk/ml_lab/research
MLFLOW_DIR: /home/zhuk/ml_lab/mlflow


In [7]:
 
import sklearn
 
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("mlflow:", mlflow.__version__)
print("optuna:", optuna.__version__)

numpy: 1.26.4
pandas: 2.3.3
sklearn: 1.7.2
mlflow: 2.16.0
optuna: 5.0.0


In [9]:
# 4. ЗАГРУЗКА ОЧИЩЕННОГО ДАТАСЕТА ИЗ ЛР1
# ============================================================
# ЗАГРУЗКА И ПОВТОРНАЯ ОЧИСТКА ДАННЫХ ИЗ ЛР1
# ============================================================
 
SOURCE_DATA_PATH = DATA_DIR / "train.csv"
 
df = pd.read_csv(SOURCE_DATA_PATH)
 
print("Исходный размер:", df.shape)
 
# ------------------------------------------------------------
# Удаляем полные дубликаты
# ------------------------------------------------------------
 
df = df.drop_duplicates().copy()
 
# ------------------------------------------------------------
# Удаляем невалидные значения размеров экрана
# Так же, как делали в ЛР1
# ------------------------------------------------------------
 
df = df[
    (df["px_height"] > 0) &
    (df["px_width"] > 0) &
    (df["sc_h"] > 0) &
    (df["sc_w"] > 0)
].copy()
 
# ------------------------------------------------------------
# Категориальные бинарные признаки
# ------------------------------------------------------------
 
categorical_features = [
    "blue",
    "dual_sim",
    "four_g",
    "three_g",
    "touch_screen",
    "wifi"
]
 
for column in categorical_features:
    df[column] = df[column].astype("int8")
 
# ------------------------------------------------------------
# Создаём признак из ЛР1
# ------------------------------------------------------------
 
df["pixel_area"] = (
    df["px_height"] *
    df["px_width"]
)
 
# ------------------------------------------------------------
# Проверка результата
# ------------------------------------------------------------
 
print("Размер после очистки:", df.shape)
print()
print("Пропущенных значений:", df.isna().sum().sum())
print("Дубликатов:", df.duplicated().sum())
 
display(df.head())
 
# ============================================================
# ПЕРЕСОХРАНЯЕМ CLEAN DATASET
# ============================================================
 
df.to_pickle(DATA_PATH)
 
print("Новый clean_dataset.pkl сохранён:")
print(DATA_PATH)
 
# ============================================================
# ПРОВЕРКА НОВОГО PICKLE
# ============================================================
 
test_df = pd.read_pickle(DATA_PATH)
 
print("Успешно загружен")
print(test_df.shape)
 
display(test_df.head())

Исходный размер: (2000, 21)
Размер после очистки: (1819, 22)

Пропущенных значений: 0
Дубликатов: 0


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range,pixel_area
0,842,0,2.2,0,1,0,7,0.6,188,2,...,756,2549,9,7,19,0,0,1,1,15120
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,1988,2631,17,3,7,1,1,0,2,1799140
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1716,2603,11,2,9,1,1,0,2,2167308
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1786,2769,16,8,11,1,0,0,2,2171776
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1212,1411,8,2,15,1,1,0,1,1464096


Новый clean_dataset.pkl сохранён:
/home/zhuk/ml_lab/data/clean_dataset.pkl
Успешно загружен
(1819, 22)


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range,pixel_area
0,842,0,2.2,0,1,0,7,0.6,188,2,...,756,2549,9,7,19,0,0,1,1,15120
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,1988,2631,17,3,7,1,1,0,2,1799140
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1716,2603,11,2,9,1,1,0,2,2167308
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1786,2769,16,8,11,1,0,0,2,2171776
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1212,1411,8,2,15,1,1,0,1,1464096


In [11]:
# 4. ЗАГРУЗКА ОЧИЩЕННОГО ДАТАСЕТА ИЗ ЛР1 
df = pd.read_pickle(DATA_PATH)
 
print("Размер датасета:", df.shape)
display(df.head())

Размер датасета: (1819, 22)


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range,pixel_area
0,842,0,2.2,0,1,0,7,0.6,188,2,...,756,2549,9,7,19,0,0,1,1,15120
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,1988,2631,17,3,7,1,1,0,2,1799140
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1716,2603,11,2,9,1,1,0,2,2167308
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1786,2769,16,8,11,1,0,0,2,2171776
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1212,1411,8,2,15,1,1,0,1,1464096


In [12]:
# 5. ПРОВЕРКА НАЛИЧИЯ PIXEL_AREA
if "pixel_area" not in df.columns:
    df["pixel_area"] = df["px_height"] * df["px_width"]
    print("Создан признак pixel_area")
else:
    print("pixel_area уже присутствует")

pixel_area уже присутствует


In [13]:
# 6. ИНФОРМАЦИЯ О ДАННЫХ
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1819 entries, 0 to 1999
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   battery_power  1819 non-null   int64  
 1   blue           1819 non-null   int8   
 2   clock_speed    1819 non-null   float64
 3   dual_sim       1819 non-null   int8   
 4   fc             1819 non-null   int64  
 5   four_g         1819 non-null   int8   
 6   int_memory     1819 non-null   int64  
 7   m_dep          1819 non-null   float64
 8   mobile_wt      1819 non-null   int64  
 9   n_cores        1819 non-null   int64  
 10  pc             1819 non-null   int64  
 11  px_height      1819 non-null   int64  
 12  px_width       1819 non-null   int64  
 13  ram            1819 non-null   int64  
 14  sc_h           1819 non-null   int64  
 15  sc_w           1819 non-null   int64  
 16  talk_time      1819 non-null   int64  
 17  three_g        1819 non-null   int8   
 18  touch_screen 

In [14]:
# 7. ЦЕЛЕВАЯ ПЕРЕМЕННАЯ
 
TARGET = "price_range"
 
print("Распределение target:")
print(df[TARGET].value_counts().sort_index())
 

Распределение target:
price_range
0    451
1    451
2    456
3    461
Name: count, dtype: int64


In [15]:
# 8. ФОРМИРОВАНИЕ X И y
 
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()
 
print("X:", X.shape)
print("y:", y.shape)

X: (1819, 21)
y: (1819,)


In [16]:
# 9. ЧИСЛОВЫЕ И КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ
 
categorical_features = [
    "blue",
    "dual_sim",
    "four_g",
    "three_g",
    "touch_screen",
    "wifi"
]
 
numeric_features = [
    column
    for column in X.columns
    if column not in categorical_features
]
 
print("Числовые признаки:")
print(numeric_features)
 
print("\nКатегориальные признаки:")
print(categorical_features)
 
print("\nКоличество числовых:", len(numeric_features))
print("Количество категориальных:", len(categorical_features))
 

Числовые признаки:
['battery_power', 'clock_speed', 'fc', 'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height', 'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'pixel_area']

Категориальные признаки:
['blue', 'dual_sim', 'four_g', 'three_g', 'touch_screen', 'wifi']

Количество числовых: 15
Количество категориальных: 6


In [17]:
# 10. TRAIN / TEST 75% / 25%
 
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)
 
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1364, 21)
X_test: (455, 21)
y_train: (1364,)
y_test: (455,)


In [18]:
# 11. ФУНКЦИЯ РАСЧЁТА МЕТРИК
 
def calculate_metrics(model, X_data, y_true):
    predictions = model.predict(X_data)
    probabilities = model.predict_proba(X_data)
 
    metrics = {
        "precision": precision_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
 
        "recall": recall_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
 
        "f1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
 
        "roc_auc": roc_auc_score(
            y_true,
            probabilities,
            multi_class="ovr",
            average="macro"
        )
    }
 
    return metrics
 

In [19]:
# 12. BASELINE TRANSFORMER
# numeric -> StandardScaler
# categorical -> TargetEncoder
 
baseline_transformer = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
 
        (
            "categorical",
            TargetEncoder(
                target_type="multiclass",
                random_state=42
            ),
            categorical_features
        )
    ],
    remainder="drop"
)

In [20]:
# 13. BASELINE PIPELINE
 
baseline_pipeline = Pipeline(
    steps=[
        (
            "transform",
            baseline_transformer
        ),
 
        (
            "classification",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)
 
baseline_pipeline

,steps,"[('transform', ...), ('classification', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [21]:
# 14. ОБУЧЕНИЕ BASELINE
 
baseline_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('transform', ...), ('classification', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [22]:
# 15. BASELINE METRICS
 
baseline_metrics = calculate_metrics(
    baseline_pipeline,
    X_test,
    y_test
)
 
print("Baseline metrics:")
print(baseline_metrics)

Baseline metrics:
{'precision': 0.8709783552304957, 'recall': 0.8724948191949671, 'f1': 0.8698749032602096, 'roc_auc': 0.9808580101925577}


In [23]:
# 16. BASELINE METRICS ТАБЛИЦЕЙ
 
baseline_metrics_df = pd.DataFrame(
    baseline_metrics.items(),
    columns=["metric", "value"]
)
 
display(baseline_metrics_df)

,metric,value
0,precision,0.870978
1,recall,0.872495
2,f1,0.869875
3,roc_auc,0.980858


In [24]:
# 17. НАСТРОЙКА MLFLOW
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
 
EXPERIMENT_NAME = "Mobile Price Classification Lab 2"
 
MODEL_NAME = "mobile-price-classifier"
 
mlflow.set_tracking_uri(
    MLFLOW_TRACKING_URI
)
 
mlflow.set_registry_uri(
    MLFLOW_TRACKING_URI
)
 
experiment = mlflow.set_experiment(
    EXPERIMENT_NAME
)
 
print(
    "Tracking URI:",
    mlflow.get_tracking_uri()
)
 
print(
    "Experiment ID:",
    experiment.experiment_id
)

2026/09/29 02:13:26 INFO mlflow.tracking.fluent: Experiment with name 'Mobile Price Classification Lab 2' does not exist. Creating a new experiment.


Tracking URI: http://127.0.0.1:5000
Experiment ID: 1


In [25]:
# 18. ПРОВЕРКА СОЕДИНЕНИЯ С MLFLOW
 
client = MlflowClient()
 
experiments = client.search_experiments()
 
for experiment in experiments:
    print(
        experiment.experiment_id,
        experiment.name
    )

1 Mobile Price Classification Lab 2
0 Default


In [26]:
# 19. ФУНКЦИЯ ЛОГИРОВАНИЯ ЭКСПЕРИМЕНТА
 
def log_model_run(
    run_name,
    model,
    metrics=None,
    params=None,
    artifacts=None,
    register_model=False,
    X_signature=None
):
    if X_signature is None:
        X_signature = X_train
 
    input_example = X_signature.head(5).copy()
 
    prediction_example = model.predict(
        input_example
    )
 
    signature = infer_signature(
        model_input=input_example,
        model_output=prediction_example
    )
 
    with mlflow.start_run(
        run_name=run_name
    ) as run:
 
        if metrics:
            mlflow.log_metrics(
                metrics
            )
 
        if params:
            mlflow.log_params(
                params
            )
 
        if REQUIREMENTS_PATH.exists():
            mlflow.log_artifact(
                str(REQUIREMENTS_PATH)
            )
 
        if artifacts:
            for artifact in artifacts:
                artifact = Path(artifact)
 
                if artifact.exists():
                    mlflow.log_artifact(
                        str(artifact)
                    )
 
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
            registered_model_name=(
                MODEL_NAME
                if register_model
                else None
            )
        )
 
        run_id = run.info.run_id
 
    return run_id

In [27]:
# 20. ЛОГИРОВАНИЕ BASELINE + VERSION 1
 
baseline_run_id = log_model_run(
    run_name="01_baseline_random_forest",
 
    model=baseline_pipeline,
 
    metrics=baseline_metrics,
 
    params={
        "model": "RandomForestClassifier",
        "n_estimators": 100,
        "random_state": 42
    },
 
    register_model=True
)
 
print(
    "Baseline run_id:",
    baseline_run_id
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'mobile-price-classifier'.
2026/09/29 02:14:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile-price-classifier, v

2026/09/29 02:14:45 INFO mlflow.tracking._tracking_service.client: 🏃 View run 01_baseline_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/9c6ac07a53674265b97b98cf516f94b1.
2026/09/29 02:14:45 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Baseline run_id: 9c6ac07a53674265b97b98cf516f94b1


In [28]:
# 21. FEATURE ENGINEERING
# ВЫБИРАЕМ ПРИЗНАКИ
 
poly_features = [
    "ram",
    "battery_power",
    "pixel_area"
]
 
bin_features = [
    "ram",
    "battery_power",
    "pixel_area"
]
 
print(
    "Polynomial features:",
    poly_features
)
 
print(
    "Binned features:",
    bin_features
)

Polynomial features: ['ram', 'battery_power', 'pixel_area']
Binned features: ['ram', 'battery_power', 'pixel_area']


In [29]:
# 22. POLYNOMIAL FEATURES
 
polynomial_pipeline = Pipeline(
    steps=[
        (
            "poly",
            PolynomialFeatures(
                degree=2,
                include_bias=False
            )
        ),
 
        (
            "scale",
            StandardScaler()
        )
    ]
)

In [30]:
# 23. KBINS
 
bins_pipeline = Pipeline(
    steps=[
        (
            "bins",
            KBinsDiscretizer(
                n_bins=5,
                encode="ordinal",
                strategy="quantile",
                subsample=None
            )
        ),
 
        (
            "scale",
            StandardScaler()
        )
    ]
)

In [31]:
# 24. РАСШИРЕННЫЙ COLUMN TRANSFORMER
 
extended_transformer = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
 
        (
            "polynomial",
            polynomial_pipeline,
            poly_features
        ),
 
        (
            "bins",
            bins_pipeline,
            bin_features
        ),
 
        (
            "categorical",
            TargetEncoder(
                target_type="multiclass",
                random_state=42
            ),
            categorical_features
        )
    ],
    remainder="drop"
)
 
extended_transformer

,transformers,"[('numeric', ...), ('polynomial', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [32]:
# 25. X_train_fe_sklearn
# ============================================================
 
X_train_fe_sklearn = (
    extended_transformer.fit_transform(
        X_train.copy(),
        y_train
    )
)
 
print(
    "Исходное количество признаков:",
    X_train.shape[1]
)
 
print(
    "После Feature Engineering:",
    X_train_fe_sklearn.shape[1]
)
 

Исходное количество признаков: 21
После Feature Engineering: 51


/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [33]:
# 26. НАЗВАНИЯ НОВЫХ ПРИЗНАКОВ
 
feature_names = (
    extended_transformer
    .get_feature_names_out()
)
 
print(
    "Количество признаков:",
    len(feature_names)
)
 
for index, name in enumerate(
    feature_names
):
    print(
        index,
        name
    )

Количество признаков: 51
0 numeric__battery_power
1 numeric__clock_speed
2 numeric__fc
3 numeric__int_memory
4 numeric__m_dep
5 numeric__mobile_wt
6 numeric__n_cores
7 numeric__pc
8 numeric__px_height
9 numeric__px_width
10 numeric__ram
11 numeric__sc_h
12 numeric__sc_w
13 numeric__talk_time
14 numeric__pixel_area
15 polynomial__ram
16 polynomial__battery_power
17 polynomial__pixel_area
18 polynomial__ram^2
19 polynomial__ram battery_power
20 polynomial__ram pixel_area
21 polynomial__battery_power^2
22 polynomial__battery_power pixel_area
23 polynomial__pixel_area^2
24 bins__ram
25 bins__battery_power
26 bins__pixel_area
27 categorical__blue_0
28 categorical__blue_1
29 categorical__blue_2
30 categorical__blue_3
31 categorical__dual_sim_0
32 categorical__dual_sim_1
33 categorical__dual_sim_2
34 categorical__dual_sim_3
35 categorical__four_g_0
36 categorical__four_g_1
37 categorical__four_g_2
38 categorical__four_g_3
39 categorical__three_g_0
40 categorical__three_g_1
41 categorical__thr

In [34]:
# 27. СОХРАНЯЕМ НАЗВАНИЯ ПРИЗНАКОВ
 
FEATURE_NAMES_PATH = (
    RESEARCH_DIR /
    "sklearn_feature_names.txt"
)
 
with open(
    FEATURE_NAMES_PATH,
    "w",
    encoding="utf-8"
) as file:
 
    for name in feature_names:
        file.write(
            f"{name}\n"
        )
 
print(
    FEATURE_NAMES_PATH
)

/home/zhuk/ml_lab/research/sklearn_feature_names.txt


In [35]:
# 28. PIPELINE С FEATURE ENGINEERING
 
feature_pipeline = Pipeline(
    steps=[
        (
            "transform",
            extended_transformer
        ),
 
        (
            "classification",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)
 
feature_pipeline

,steps,"[('transform', ...), ('classification', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('polynomial', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [36]:
# 29. ОБУЧЕНИЕ FEATURE ENGINEERING MODEL
 
feature_pipeline.fit(
    X_train,
    y_train
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


,steps,"[('transform', ...), ('classification', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('polynomial', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [37]:
# 30. МЕТРИКИ FEATURE ENGINEERING
 
feature_metrics = calculate_metrics(
    feature_pipeline,
    X_test,
    y_test
)
 
print(
    "Feature Engineering metrics:"
)
 
print(
    feature_metrics
)

Feature Engineering metrics:
{'precision': 0.9527826118839666, 'recall': 0.9516566763195021, 'f1': 0.9514442537076875, 'roc_auc': 0.9962985260951894}


In [38]:
# 31. ЛОГИРОВАНИЕ FEATURE ENGINEERING
 
feature_run_id = log_model_run(
    run_name="02_sklearn_feature_engineering",
 
    model=feature_pipeline,
 
    metrics=feature_metrics,
 
    params={
        "polynomial_degree": 2,
        "kbins": 5,
        "n_estimators": 100
    },
 
    artifacts=[
        FEATURE_NAMES_PATH
    ]
)
 
print(
    "Feature Engineering run_id:",
    feature_run_id
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/09/29 02:17:43 INFO mlflow.tracking._tracking_service.client: 🏃 View run 02_sklearn_feature_engineering at: http://127.0.0.1:5000/#/experiments/1/runs/809634ce561c4038b9bde7d064842318.
2026/09/29 02:17:43 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Feature Engineering run_id: 809634ce561c4038b9bde7d064842318


In [39]:
# 32. FEATURE SELECTION
 
selection_keywords = [
    "ram",
    "battery_power",
    "pixel_area",
    "px_height",
    "px_width"
]
 
selected_indices = [
    index
    for index, name
    in enumerate(feature_names)
 
    if any(
        keyword in name
        for keyword in selection_keywords
    )
]
 
selected_feature_names = [
    feature_names[index]
    for index in selected_indices
]
 
print(
    "Всего после Feature Engineering:",
    len(feature_names)
)
 
print(
    "Отобрано:",
    len(selected_feature_names)
)
 
selected_ratio = (
    len(selected_feature_names)
    /
    len(feature_names)
)
 
print(
    "Доля:",
    selected_ratio
)
 
for index, name in zip(
    selected_indices,
    selected_feature_names
):
    print(
        index,
        name
    )

Всего после Feature Engineering: 51
Отобрано: 17
Доля: 0.3333333333333333
0 numeric__battery_power
8 numeric__px_height
9 numeric__px_width
10 numeric__ram
14 numeric__pixel_area
15 polynomial__ram
16 polynomial__battery_power
17 polynomial__pixel_area
18 polynomial__ram^2
19 polynomial__ram battery_power
20 polynomial__ram pixel_area
21 polynomial__battery_power^2
22 polynomial__battery_power pixel_area
23 polynomial__pixel_area^2
24 bins__ram
25 bins__battery_power
26 bins__pixel_area


In [40]:
# 33. ПРОВЕРКА ДОЛИ ПРИЗНАКОВ
# ============================================================
 
assert (
    0.20
    <= selected_ratio
    <= 0.70
), (
    f"Выбрано "
    f"{selected_ratio:.2%} "
    "признаков. "
    "Нужно ориентировочно "
    "от 20% до 70%."
)
 
print(
    f"Выбрано "
    f"{selected_ratio:.2%} "
    "признаков"
)

Выбрано 33.33% признаков


In [41]:
# 34. СОХРАНЯЕМ ИНДЕКСЫ И ИМЕНА
# ============================================================
 
SELECTED_INDICES_PATH = (
    RESEARCH_DIR /
    "selected_indices.txt"
)
 
SELECTED_FEATURES_PATH = (
    RESEARCH_DIR /
    "selected_features.txt"
)
 
with open(
    SELECTED_INDICES_PATH,
    "w",
    encoding="utf-8"
) as file:
 
    for index in selected_indices:
        file.write(
            f"{index}\n"
        )
 
with open(
    SELECTED_FEATURES_PATH,
    "w",
    encoding="utf-8"
) as file:
 
    for name in selected_feature_names:
        file.write(
            f"{name}\n"
        )
 
print(
    SELECTED_INDICES_PATH
)
 
print(
    SELECTED_FEATURES_PATH
)

/home/zhuk/ml_lab/research/selected_indices.txt
/home/zhuk/ml_lab/research/selected_features.txt


In [42]:
# 35. CUSTOM TRANSFORMER ДЛЯ FEATURE SELECTION
# ============================================================
 
class ColumnIndexSelector(
    BaseEstimator,
    TransformerMixin
):
 
    def __init__(
        self,
        indices
    ):
        self.indices = indices
 
    def fit(
        self,
        X,
        y=None
    ):
        return self
 
    def transform(
        self,
        X
    ):
        return X[
            :,
            self.indices
        ]

In [43]:
# 36. PIPELINE С FEATURE SELECTION
# ============================================================
 
selected_pipeline = Pipeline(
    steps=[
        (
            "transform",
            extended_transformer
        ),
 
        (
            "feature_selection",
            ColumnIndexSelector(
                selected_indices
            )
        ),
 
        (
            "classification",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)
 
selected_pipeline

,steps,"[('transform', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('polynomial', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [44]:
# 37. ОБУЧЕНИЕ FEATURE SELECTION
# ============================================================
 
selected_pipeline.fit(
    X_train,
    y_train
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


,steps,"[('transform', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('polynomial', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [45]:
# 38. МЕТРИКИ FEATURE SELECTION
# ============================================================
 
selected_metrics = calculate_metrics(
    selected_pipeline,
    X_test,
    y_test
)
 
print(
    "Feature Selection metrics:"
)
 
print(
    selected_metrics
)

Feature Selection metrics:
{'precision': 0.9477930844408117, 'recall': 0.9473091877442742, 'f1': 0.9469387212592915, 'roc_auc': 0.9960792399369045}


In [46]:
# 39. ЛОГИРОВАНИЕ FEATURE SELECTION
# ============================================================
 
selected_run_id = log_model_run(
    run_name="03_feature_selection",
 
    model=selected_pipeline,
 
    metrics=selected_metrics,
 
    params={
        "selected_features_count":
            len(selected_indices),
 
        "selected_features_ratio":
            selected_ratio
    },
 
    artifacts=[
        SELECTED_INDICES_PATH,
        SELECTED_FEATURES_PATH
    ]
)
 
print(
    "Feature Selection run_id:",
    selected_run_id
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/09/29 02:19:12 INFO mlflow.tracking._tracking_service.client: 🏃 View run 03_feature_selection at: http://127.0.0.1:5000/#/experiments/1/runs/7a7fa8df05c54d38a4f49a9d08cb8b02.
2026/09/29 02:19:12 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Feature Selection run_id: 7a7fa8df05c54d38a4f49a9d08cb8b02


In [47]:
# 40. СРАВНЕНИЕ МОДЕЛЕЙ ДО OPTUNA
# ============================================================
 
pre_optuna_results = pd.DataFrame([
    {
        "model": "baseline",
        **baseline_metrics
    },
 
    {
        "model":
            "feature_engineering",
        **feature_metrics
    },
 
    {
        "model":
            "feature_selection",
        **selected_metrics
    }
])
 
pre_optuna_results = (
    pre_optuna_results
    .sort_values(
        "f1",
        ascending=False
    )
)
 
display(
    pre_optuna_results
)

,model,precision,recall,f1,roc_auc
1,feature_engineering,0.952783,0.951657,0.951444,0.996299
2,feature_selection,0.947793,0.947309,0.946939,0.996079
0,baseline,0.870978,0.872495,0.869875,0.980858


In [48]:
# 41. ВЫБИРАЕМ ЛУЧШУЮ МОДЕЛЬ ДО OPTUNA
# ============================================================
 
pre_optuna_models = {
    "baseline":
        baseline_pipeline,
 
    "feature_engineering":
        feature_pipeline,
 
    "feature_selection":
        selected_pipeline
}
 
pre_optuna_metrics = {
    "baseline":
        baseline_metrics,
 
    "feature_engineering":
        feature_metrics,
 
    "feature_selection":
        selected_metrics
}
 
best_pre_optuna_name = max(
    pre_optuna_metrics,
 
    key=lambda name:
        pre_optuna_metrics[
            name
        ]["f1"]
)
 
best_pre_optuna_template = (
    pre_optuna_models[
        best_pre_optuna_name
    ]
)
 
print(
    "Лучшая модель до Optuna:",
    best_pre_optuna_name
)
 
print(
    "F1:",
    pre_optuna_metrics[
        best_pre_optuna_name
    ]["f1"]
)

Лучшая модель до Optuna: feature_engineering
F1: 0.9514442537076875


In [49]:
# 42. CROSS VALIDATION ДЛЯ OPTUNA
# ============================================================
 
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [50]:
# 43. OPTUNA OBJECTIVE
# ============================================================
 
def objective(trial):
 
    n_estimators = (
        trial.suggest_int(
            "n_estimators",
            50,
            200
        )
    )
 
    max_depth = (
        trial.suggest_int(
            "max_depth",
            3,
            25
        )
    )
 
    max_features = (
        trial.suggest_float(
            "max_features",
            0.1,
            1.0
        )
    )
 
    trial_model = clone(
        best_pre_optuna_template
    )
 
    trial_model.set_params(
        classification__n_estimators=
            n_estimators,
 
        classification__max_depth=
            max_depth,
 
        classification__max_features=
            max_features,
 
        classification__random_state=
            42,
 
        classification__n_jobs=
            -1
    )
 
    scores = cross_val_score(
        trial_model,
        X_train,
        y_train,
 
        scoring="f1_macro",
 
        cv=cv,
 
        n_jobs=-1
    )
 
    return scores.mean()

In [51]:
# 44. СОЗДАЁМ OPTUNA STUDY
# F1 НУЖНО МАКСИМИЗИРОВАТЬ
# ============================================================
 
study = optuna.create_study(
    direction="maximize",
    study_name="random_forest_f1"
)
 
print(
    "Direction:",
    study.direction
)

[I 2026-09-29 02:20:15,370] A new study created in memory with name: random_forest_f1


Direction: StudyDirection.MAXIMIZE


In [53]:
# 45. 10 OPTUNA TRIALS
# ============================================================
 
study.optimize(
    objective,
    n_trials=10
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current def

In [54]:
# 46. ЛУЧШИЕ ПАРАМЕТРЫ OPTUNA
# ============================================================
 
print(
    "Best CV F1:"
)
 
print(
    study.best_value
)
 
print(
    "\nBest parameters:"
)
 
print(
    study.best_params
)

Best CV F1:
0.9037183007679435

Best parameters:
{'n_estimators': 137, 'max_depth': 17, 'max_features': 0.19570087435374373}


In [55]:
# 47. ТАБЛИЦА OPTUNA TRIALS
# ============================================================
 
optuna_trials = (
    study.trials_dataframe()
)
 
display(
    optuna_trials[
        [
            "number",
            "value",
            "params_n_estimators",
            "params_max_depth",
            "params_max_features",
            "state"
        ]
    ]
)

,number,value,params_n_estimators,params_max_depth,params_max_features,state
0,0,0.899995,89,25,0.280640,COMPLETE
1,1,0.892962,179,21,0.587772,COMPLETE
2,2,0.886371,91,9,0.580549,COMPLETE
3,3,0.901536,125,18,0.173013,COMPLETE
4,4,0.881686,152,6,0.630771,COMPLETE
5,5,0.888194,167,18,0.871399,COMPLETE
6,6,0.897840,191,19,0.480583,COMPLETE
7,7,0.892821,64,16,0.484663,COMPLETE
8,8,0.893349,59,14,0.279771,COMPLETE
9,9,0.889652,200,18,0.848368,COMPLETE


In [56]:
# 48. СОХРАНЯЕМ OPTUNA TRIALS
# ============================================================
 
OPTUNA_RESULTS_PATH = (
    RESEARCH_DIR /
    "optuna_trials.txt"
)
 
with open(
    OPTUNA_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as file:
 
    file.write(
        optuna_trials.to_string(
            index=False
        )
    )
 
print(
    OPTUNA_RESULTS_PATH
)

/home/zhuk/ml_lab/research/optuna_trials.txt


In [57]:
# 49. СОЗДАЁМ TUNED PIPELINE
# ============================================================
 
tuned_pipeline = clone(
    best_pre_optuna_template
)
 
tuned_pipeline.set_params(
 
    classification__n_estimators=
        study.best_params[
            "n_estimators"
        ],
 
    classification__max_depth=
        study.best_params[
            "max_depth"
        ],
 
    classification__max_features=
        study.best_params[
            "max_features"
        ],
 
    classification__random_state=
        42,
 
    classification__n_jobs=
        -1
)
 
tuned_pipeline

,steps,"[('transform', ...), ('classification', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('polynomial', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [58]:
# 50. ОБУЧАЕМ TUNED MODEL
# ============================================================
 
tuned_pipeline.fit(
    X_train,
    y_train
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


,steps,"[('transform', ...), ('classification', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('polynomial', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [59]:
# 51. МЕТРИКИ TUNED MODEL
# ============================================================
 
tuned_metrics = calculate_metrics(
    tuned_pipeline,
    X_test,
    y_test
)
 
print(
    "Optuna metrics:"
)
 
print(
    tuned_metrics
)

Optuna metrics:
{'precision': 0.9568360301314847, 'recall': 0.9560235718191207, 'f1': 0.9558031022411555, 'roc_auc': 0.996523367207754}


In [60]:
# 52. ЛОГИРУЕМ OPTUNA MODEL
# И РЕГИСТРИРУЕМ НОВУЮ VERSION
# ============================================================
 
tuned_run_id = log_model_run(
 
    run_name="04_optuna_tuned",
 
    model=tuned_pipeline,
 
    metrics=tuned_metrics,
 
    params={
        "base_pipeline":
            best_pre_optuna_name,
 
        **study.best_params
    },
 
    artifacts=[
        OPTUNA_RESULTS_PATH
    ],
 
    register_model=True
)
 
print(
    "Optuna run_id:",
    tuned_run_id
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'mobile-price-classifier' already exists. Creating a new version of this model...
2026/09/29 02:22:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creati

2026/09/29 02:22:17 INFO mlflow.tracking._tracking_service.client: 🏃 View run 04_optuna_tuned at: http://127.0.0.1:5000/#/experiments/1/runs/d0575f485ab24dcfbd97807548dc3d2d.
2026/09/29 02:22:17 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Optuna run_id: d0575f485ab24dcfbd97807548dc3d2d


In [61]:
# 53. ИТОГОВОЕ СРАВНЕНИЕ ВСЕХ ЭКСПЕРИМЕНТОВ
# ============================================================
 
results = pd.DataFrame([
    {
        "model": "baseline",
        **baseline_metrics
    },
 
    {
        "model":
            "feature_engineering",
        **feature_metrics
    },
 
    {
        "model":
            "feature_selection",
        **selected_metrics
    },
 
    {
        "model": "optuna",
        **tuned_metrics
    }
])
 
results = (
    results
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)
 
display(
    results
)

,model,precision,recall,f1,roc_auc
0,optuna,0.956836,0.956024,0.955803,0.996523
1,feature_engineering,0.952783,0.951657,0.951444,0.996299
2,feature_selection,0.947793,0.947309,0.946939,0.996079
3,baseline,0.870978,0.872495,0.869875,0.980858


In [62]:
# 54. СОХРАНЯЕМ РЕЗУЛЬТАТЫ
# ============================================================
 
RESULTS_PATH = (
    RESEARCH_DIR /
    "results.txt"
)
 
with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as file:
 
    file.write(
        results.to_string(
            index=False
        )
    )
 
print(
    RESULTS_PATH
)

/home/zhuk/ml_lab/research/results.txt


In [63]:
# 55. ОПРЕДЕЛЯЕМ ЛУЧШУЮ МОДЕЛЬ
# ============================================================
 
all_models = {
 
    "baseline":
        baseline_pipeline,
 
    "feature_engineering":
        feature_pipeline,
 
    "feature_selection":
        selected_pipeline,
 
    "optuna":
        tuned_pipeline
}
 
all_metrics = {
 
    "baseline":
        baseline_metrics,
 
    "feature_engineering":
        feature_metrics,
 
    "feature_selection":
        selected_metrics,
 
    "optuna":
        tuned_metrics
}
 
best_model_name = max(
    all_metrics,
 
    key=lambda name:
        all_metrics[
            name
        ]["f1"]
)
 
best_model_template = (
    all_models[
        best_model_name
    ]
)
 
best_model_metrics = (
    all_metrics[
        best_model_name
    ]
)
 
print(
    "Лучшая модель:",
    best_model_name
)
 
print(
    "Метрики:",
    best_model_metrics
)

Лучшая модель: optuna
Метрики: {'precision': 0.9568360301314847, 'recall': 0.9560235718191207, 'f1': 0.9558031022411555, 'roc_auc': 0.996523367207754}


In [64]:
# 56. СОЗДАЁМ PRODUCTION MODEL
# ============================================================
 
production_model = clone(
    best_model_template
)
 
production_model

,steps,"[('transform', ...), ('classification', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('polynomial', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [65]:
# 57. ОБУЧАЕМ PRODUCTION MODEL НА ВСЕЙ ВЫБОРКЕ
# ============================================================
 
production_model.fit(
    X,
    y
)
 
print(
    "Production model обучена "
    "на всей выборке:",
    X.shape
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


Production model обучена на всей выборке: (1819, 21)


In [66]:
# 58. СОХРАНЯЕМ СПИСОК ИСХОДНЫХ КОЛОНОК
# ============================================================
 
PRODUCTION_FEATURES_PATH = (
    RESEARCH_DIR /
    "production_features.txt"
)
 
with open(
    PRODUCTION_FEATURES_PATH,
    "w",
    encoding="utf-8"
) as file:
 
    for column in X.columns:
        file.write(
            f"{column}\n"
        )
 
print(
    PRODUCTION_FEATURES_PATH
)

/home/zhuk/ml_lab/research/production_features.txt


In [67]:
# 59. ПАРАМЕТРЫ PRODUCTION
# ============================================================
 
production_params = {
    "source_model":
        best_model_name
}
 
if best_model_name == "optuna":
 
    production_params.update(
        study.best_params
    )
 
print(
    production_params
)

{'source_model': 'optuna', 'n_estimators': 137, 'max_depth': 17, 'max_features': 0.19570087435374373}


In [68]:
# 60. ЛОГИРУЕМ PRODUCTION MODEL
# ============================================================
 
input_example = (
    X.head(5).copy()
)
 
production_signature = infer_signature(
 
    model_input=
        input_example,
 
    model_output=
        production_model.predict(
            input_example
        )
)
 
with mlflow.start_run(
    run_name=
        "05_production_full_dataset"
) as run:
 
    mlflow.log_params(
        production_params
    )
 
    if REQUIREMENTS_PATH.exists():
 
        mlflow.log_artifact(
            str(
                REQUIREMENTS_PATH
            )
        )
 
    mlflow.log_artifact(
        str(
            PRODUCTION_FEATURES_PATH
        )
    )
 
    mlflow.log_artifact(
        str(
            RESULTS_PATH
        )
    )
 
    mlflow.sklearn.log_model(
 
        sk_model=
            production_model,
 
        artifact_path=
            "model",
 
        signature=
            production_signature,
 
        input_example=
            input_example,
 
        registered_model_name=
            MODEL_NAME
    )
 
    production_run_id = (
        run.info.run_id
    )
 
print(
    "Production run_id:",
    production_run_id
)

/home/zhuk/ml_lab/.venv_ml_lab/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'mobile-price-classifier' already exists. Creating a new version of this model...
2026/09/29 02:23:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creati

2026/09/29 02:23:39 INFO mlflow.tracking._tracking_service.client: 🏃 View run 05_production_full_dataset at: http://127.0.0.1:5000/#/experiments/1/runs/d804173f8a1949c99c5538419e7cccca.
2026/09/29 02:23:39 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Production run_id: d804173f8a1949c99c5538419e7cccca


In [69]:
# 61. ПРОВЕРЯЕМ MODEL REGISTRY
# ============================================================
 
client = MlflowClient()
 
versions = (
    client.search_model_versions(
        f"name='{MODEL_NAME}'"
    )
)
 
for version in sorted(
    versions,
    key=lambda v:
        int(v.version)
):
 
    print(
        "version:",
        version.version,
 
        "| run_id:",
        version.run_id,
 
        "| status:",
        version.status
    )

version: 1 | run_id: 9c6ac07a53674265b97b98cf516f94b1 | status: READY
version: 2 | run_id: d0575f485ab24dcfbd97807548dc3d2d | status: READY
version: 3 | run_id: d804173f8a1949c99c5538419e7cccca | status: READY


In [70]:
# 62. НАХОДИМ VERSION PRODUCTION RUN
# ============================================================
 
production_version = None
 
for version in versions:
 
    if (
        version.run_id
        ==
        production_run_id
    ):
        production_version = version
        break
 
if production_version is None:
 
    raise RuntimeError(
        "Версия Production-модели "
        "в Registry не найдена"
    )
 
print(
    "Production model version:",
    production_version.version
)

Production model version: 3


In [71]:
# 63. УСТАНАВЛИВАЕМ TAG Production
# ============================================================
 
client.set_model_version_tag(
 
    name=
        MODEL_NAME,
 
    version=
        production_version.version,
 
    key=
        "Production",
 
    value=
        "true"
)
 
client.set_model_version_tag(
 
    name=
        MODEL_NAME,
 
    version=
        production_version.version,
 
    key=
        "source_model",
 
    value=
        best_model_name
)
 
print(
    "Production tag установлен "
    "для version",
    production_version.version
)

Production tag установлен для version 3


In [72]:
# 64. ПРОВЕРЯЕМ ТЕГИ
# ============================================================
 
model_version_info = (
    client.get_model_version(
 
        name=
            MODEL_NAME,
 
        version=
            production_version.version
    )
)
 
print(
    "Version:",
    model_version_info.version
)
 
print(
    "Run ID:",
    model_version_info.run_id
)
 
print(
    "Tags:",
    model_version_info.tags
)

Version: 3
Run ID: d804173f8a1949c99c5538419e7cccca
Tags: {'Production': 'true', 'source_model': 'optuna'}


In [73]:
# 65. СКАЧИВАЕМ MLmodel ИЗ MLFLOW
# ============================================================
 
downloaded_mlmodel = (
    client.download_artifacts(
 
        production_run_id,
 
        "model/MLmodel"
    )
)
 
print(
    downloaded_mlmodel
)

/home/zhuk/ml_lab/mlflow/mlartifacts/1/d804173f8a1949c99c5538419e7cccca/artifacts/model/MLmodel


In [74]:
# 66. КОПИРУЕМ MLmodel В research/
# ============================================================
 
MLMODEL_PATH = (
    RESEARCH_DIR /
    "MLmodel"
)
 
shutil.copy2(
    downloaded_mlmodel,
    MLMODEL_PATH
)
 
print(
    "MLmodel сохранён:",
    MLMODEL_PATH
)

MLmodel сохранён: /home/zhuk/ml_lab/research/MLmodel


In [75]:
# 67. ИТОГОВАЯ ИНФОРМАЦИЯ ДЛЯ ОТЧЁТА
# ============================================================
 
print(
    "========== ИТОГ ЛР2 =========="
)
 
print(
    "\nЛучшая модель:"
)
 
print(
    best_model_name
)
 
print(
    "\nЕё тестовые метрики:"
)
 
for name, value in (
    best_model_metrics.items()
):
 
    print(
        f"{name}: "
        f"{value:.6f}"
    )
 
print(
    "\nProduction run_id:"
)
 
print(
    production_run_id
)
 
print(
    "\nProduction version:"
)
 
print(
    production_version.version
)
 
print(
    "\nProduction params:"
)
 
for name, value in (
    production_params.items()
):
 
    print(
        name,
        "=",
        value
    )
 

========== ИТОГ ЛР2 ==========

Лучшая модель:
optuna

Её тестовые метрики:
precision: 0.956836
recall: 0.956024
f1: 0.955803
roc_auc: 0.996523

Production run_id:
d804173f8a1949c99c5538419e7cccca

Production version:
3

Production params:
source_model = optuna
n_estimators = 137
max_depth = 17
max_features = 0.19570087435374373


In [76]:
# 68. ТАБЛИЦА ДЛЯ ОТЧЁТА
# ============================================================
 
report_results = (
    results.copy()
)
 
for column in [
    "precision",
    "recall",
    "f1",
    "roc_auc"
]:
 
    report_results[column] = (
        report_results[column]
        .round(4)
    )
 
display(
    report_results
)
 

,model,precision,recall,f1,roc_auc
0,optuna,0.9568,0.9560,0.9558,0.9965
1,feature_engineering,0.9528,0.9517,0.9514,0.9963
2,feature_selection,0.9478,0.9473,0.9469,0.9961
3,baseline,0.8710,0.8725,0.8699,0.9809


In [77]:
# 69. СОХРАНЯЕМ SUMMARY ЛАБОРАТОРНОЙ
# ============================================================
 
SUMMARY_PATH = (
    RESEARCH_DIR /
    "lab2_summary.txt"
)
 
with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:
 
    file.write(
        "Лабораторная работа №2\n\n"
    )
 
    file.write(
        "Результаты экспериментов:\n"
    )
 
    file.write(
        report_results.to_string(
            index=False
        )
    )
 
    file.write(
        "\n\nЛучшая модель: "
        f"{best_model_name}\n"
    )
 
    file.write(
        "Production run_id: "
        f"{production_run_id}\n"
    )
 
    file.write(
        "Production version: "
        f"{production_version.version}\n"
    )
 
    file.write(
        "\nProduction parameters:\n"
    )
 
    for key, value in (
        production_params.items()
    ):
 
        file.write(
            f"{key}: {value}\n"
        )
 
print(
    SUMMARY_PATH
)

/home/zhuk/ml_lab/research/lab2_summary.txt


In [78]:
# 70. ПРОВЕРЯЕМ СОЗДАННЫЕ АРТЕФАКТЫ
# ============================================================
 
expected_files = [
 
    FEATURE_NAMES_PATH,
 
    SELECTED_INDICES_PATH,
 
    SELECTED_FEATURES_PATH,
 
    OPTUNA_RESULTS_PATH,
 
    RESULTS_PATH,
 
    PRODUCTION_FEATURES_PATH,
 
    MLMODEL_PATH,
 
    SUMMARY_PATH
]
 
for file_path in expected_files:
 
    print(
        "[OK]"
        if file_path.exists()
        else "[MISSING]",
 
        file_path.name
    )

[OK] sklearn_feature_names.txt
[OK] selected_indices.txt
[OK] selected_features.txt
[OK] optuna_trials.txt
[OK] results.txt
[OK] production_features.txt
[OK] MLmodel
[OK] lab2_summary.txt


In [79]:
# 71. СПИСОК ФАЙЛОВ research/
# ============================================================
 
print(
    "\nФайлы research/:"
)
 
for file_path in sorted(
    RESEARCH_DIR.iterdir()
):
 
    print(
        file_path.name,
        "-",
        round(
            file_path.stat().st_size
            /
            1024,
            2
        ),
        "KB"
    )


Файлы research/:
.ipynb_checkpoints - 4.0 KB
MLmodel - 1.99 KB
Untitled.ipynb - 749.49 KB
lab2_summary.txt - 0.57 KB
optuna_trials.txt - 4.9 KB
production_features.txt - 0.16 KB
results.txt - 0.28 KB
selected_features.txt - 0.36 KB
selected_indices.txt - 0.05 KB
sklearn_feature_names.txt - 1.06 KB
